# Camada Gold

In [0]:
CREATE SCHEMA IF NOT EXISTS projeto_olist.olist_gold;


In [0]:
-- 1. Criando dim_customers 
CREATE OR REPLACE TABLE projeto_olist.olist_gold.customers_dim AS
SELECT 
    customer_unique_id,
    customer_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM projeto_olist.olist_silver.olist_customers_silver;

-- 2. Criando dim_sellers

CREATE OR REPLACE TABLE projeto_olist.olist_gold.sellers_dim AS
SELECT 
    seller_id,
    seller_zip_code_prefix,
    seller_state,
    seller_city
FROM projeto_olist.olist_silver.olist_sellers_silver;

-- 3. Criando dim_products

CREATE OR REPLACE TABLE projeto_olist.olist_gold.products_dim AS
SELECT 
    product_id,
    product_category_name,
    product_weight_g
FROM projeto_olist.olist_silver.olist_products_silver;

-- 4. Criando dim_calendar

CREATE OR REPLACE TABLE projeto_olist.olist_gold.calendar_dim AS
WITH cte_data AS(
    SELECT EXPLODE(SEQUENCE(TO_DATE('2016-01-01'), TO_DATE('2019-12-31'),INTERVAL 1 DAY)) AS date_actual
)

SELECT 
    date_actual,
    DAY(date_actual) AS date_of_month,
    MONTH(date_actual) AS month_number,
    DATE_FORMAT(date_actual,'MMMM') AS month_name,
    YEAR(date_actual) AS year_actual,
    DATE_FORMAT(date_actual,'EEEE') AS day_of_week,
    CASE WHEN DATE_FORMAT(date_actual, 'E') IN ('Sat','Sun') THEN 1 ELSE 0 END AS is_weekend
FROM cte_data;

-- 5. Criando fact_sales

CREATE OR REPLACE TABLE projeto_olist.olist_gold.sales_fact AS
SELECT 
    oi.order_id,
    oi.order_item_id,

    oi.product_id,
    o.customer_id,
    oi.seller_id,

    o.order_purchase_timestamp AS order_purchase_date,
    o.order_status,

    oi.price,
    oi.freight_value

FROM projeto_olist.olist_silver.olist_orders_items_silver oi  
INNER JOIN projeto_olist.olist_silver.olist_orders_silver o  
ON oi.order_id = o.order_id;

